# LOADING DATA

In [1]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

Code directory: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code

Data paths:
  ✓ X_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote.parquet
  ✓ X_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_tomek.parquet
  ✓ X_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote_tomek.parquet
  ✓ y_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\smote\y_smote.pkl
  ✓ y_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\tomek\y_tomek.pkl
  ✓ y_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessi

# RF

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import random
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import os # For getting CPU count

# --- Part 1: Decision Tree Components (Unchanged) ---
# These classes remain the same, as the slow calculation is within _best_split,
# but we will now run the entire tree training in parallel.

class Node:
    """A node in the decision tree."""
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class DecisionTree:
    """A single Decision Tree Classifier."""
    def __init__(self, min_samples_split=2, max_depth=100, n_features=None, random_state=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.random_state = random_state
        self.root = None

    def _gini(self, y):
        """Calculate the Gini impurity for a set of labels."""
        if len(y) == 0:
            return 0
        counts = Counter(y)
        total_samples = len(y)
        gini = 1.0
        for label in counts:
            prob_k = counts[label] / total_samples
            gini -= prob_k**2
        return gini

    def _best_split(self, X, y, feat_indices):
        """Find the best feature and threshold to split the data."""
        best_gain = -1
        split_idx, split_thresh = None, None
        n_samples, n_features = X.shape
        current_gini = self._gini(y)

        for feat_idx in feat_indices:
            thresholds = np.unique(X[:, feat_idx])
            
            for threshold in thresholds:
                left_indices = np.where(X[:, feat_idx] <= threshold)[0]
                right_indices = np.where(X[:, feat_idx] > threshold)[0]

                if len(left_indices) > 0 and len(right_indices) > 0:
                    y_left, y_right = y[left_indices], y[right_indices]
                    
                    n_l, n_r = len(y_left), len(y_right)
                    n_total = n_samples
                    
                    gini_l = self._gini(y_left)
                    gini_r = self._gini(y_right)
                    
                    weighted_gini = (n_l / n_total) * gini_l + (n_r / n_total) * gini_r
                    information_gain = current_gini - weighted_gini
                    
                    if information_gain > best_gain:
                        best_gain = information_gain
                        split_idx = feat_idx
                        split_thresh = threshold

        return split_idx, split_thresh

    def _grow_tree(self, X, y, depth):
        """Recursively build the decision tree."""
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        # Stopping criteria
        if (n_labels == 1 or 
            depth >= self.max_depth or 
            n_samples < self.min_samples_split):
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)

        # Feature Subsetting
        n_feats = self.n_features if self.n_features else n_features
        # Ensure random sampling is done safely if no random_state is used globally
        random.seed(self.random_state + depth) if self.random_state is not None else None
        feat_indices = random.sample(range(n_features), k=n_feats)

        split_idx, split_thresh = self._best_split(X, y, feat_indices)
        
        if split_idx is None:
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)
        
        left_indices = np.where(X[:, split_idx] <= split_thresh)[0]
        right_indices = np.where(X[:, split_idx] > split_thresh)[0]
        
        left_child = self._grow_tree(X[left_indices, :], y[left_indices], depth + 1)
        right_child = self._grow_tree(X[right_indices, :], y[right_indices], depth + 1)
        
        return Node(split_idx, split_thresh, left_child, right_child)

    def fit(self, X, y):
        """Build the tree."""
        # Ensure data is numpy array before starting
        X = X.values if isinstance(X, pd.DataFrame) else X
        y = y.values if isinstance(y, pd.Series) else y
        
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)
        self.root = self._grow_tree(X, y, 0)

    def _traverse_tree(self, x, node):
        """Traverse the tree to get a single prediction."""
        if node.value is not None:
            return node.value

        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def predict(self, X):
        """Make predictions for the dataset."""
        X = X.values if isinstance(X, pd.DataFrame) else X
        return np.array([self._traverse_tree(x, self.root) for x in X])

# --- Helper function for parallel execution ---

def _train_single_tree(X, y, params, tree_seed):
    """Function executed by a worker process to train one tree."""
    # Local random state to ensure independence
    np.random.seed(tree_seed)
    random.seed(tree_seed)

    # 1. Bootstrap sample
    n_samples = X.shape[0]
    n_samples_bootstrap = int(n_samples * params['bootstrap_ratio'])
    indices = np.random.choice(n_samples, n_samples_bootstrap, replace=True)
    X_sample, y_sample = X.iloc[indices], y.iloc[indices]
    
    # 2. Train tree
    tree = DecisionTree(
        min_samples_split=params['min_samples_split'],
        max_depth=params['max_depth'],
        n_features=params['n_feats'],
        random_state=tree_seed # Pass seed to tree for deterministic growth
    )
    tree.fit(X_sample, y_sample)
    return tree

# --- Part 2: Optimized Random Forest Implementation ---

class RandomForestClassifier:
    """The Random Forest Classifier implementation with Multiprocessing."""
    def __init__(self, n_estimators=100, min_samples_split=2, max_depth=10, 
                 n_features_ratio=None, bootstrap_ratio=1.0, random_state=None, n_jobs=None):
        
        self.n_estimators = n_estimators
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features_ratio = n_features_ratio
        self.bootstrap_ratio = bootstrap_ratio
        self.random_state = random_state
        self.n_jobs = n_jobs if n_jobs is not None else os.cpu_count() # Use all available cores by default
        self.trees = []

    def fit(self, X, y):
        """Train the random forest using multiprocessing."""
        if self.random_state:
            np.random.seed(self.random_state)
            random.seed(self.random_state)
            
        self.trees = []
        n_features = X.shape[1]
        
        # Determine the number of features for each tree's split
        n_feats = int(n_features * self.n_features_ratio) if self.n_features_ratio else int(np.sqrt(n_features))
        
        print(f"Training {self.n_estimators} trees using {self.n_jobs} parallel processes...")
        print(f"  Max Depth: {self.max_depth}, Features per split: {n_feats}")

        # Parameters to pass to the helper function
        params = {
            'min_samples_split': self.min_samples_split,
            'max_depth': self.max_depth,
            'n_feats': n_feats,
            'bootstrap_ratio': self.bootstrap_ratio
        }
        
        # Use ProcessPoolExecutor for parallel training
        # We cap the maximum workers at n_estimators, or the CPU count, whichever is smaller.
        max_workers = min(self.n_estimators, self.n_jobs)
        
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            
            # Submit tasks for training all trees
            for i in range(self.n_estimators):
                # Generate a unique seed for each tree
                tree_seed = self.random_state + i if self.random_state is not None else i
                future = executor.submit(_train_single_tree, X, y, params, tree_seed)
                futures.append(future)

            # Collect results as they complete (as_completed)
            for i, future in enumerate(as_completed(futures)):
                tree = future.result()
                self.trees.append(tree)
                if (i + 1) % (self.n_estimators // 10) == 0 or i == self.n_estimators - 1:
                    print(f"  Trained {i + 1}/{self.n_estimators} trees completed.")

    def _most_common_label(self, y):
        """Helper to find the majority vote from the predictions of all trees."""
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X):
        """Get the final prediction by combining the predictions of all trees (majority vote)."""
        X = X.values if isinstance(X, pd.DataFrame) else X
        
        # Parallelize predictions on the test set as well
        # We can use ThreadPoolExecutor for prediction as it's less CPU intensive
        max_workers = self.n_jobs
        
        # Get predictions from all individual trees (still sequential, but faster than pure Python loops)
        # Prediction is usually much faster than training, so multiprocessing isn't always necessary here,
        # but for consistency and speed, we'll keep the loop over trees as efficient as possible.
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        
        tree_preds = tree_preds.T 
        
        # Aggregate predictions using majority vote
        y_pred = np.array([self._most_common_label(sample_preds) for sample_preds in tree_preds])
        
        return y_pred
    
# --- Part 3: Training and Evaluation (Unchanged) ---

def accuracy(y_true, y_pred):
    """Calculate the accuracy score."""
    y_true = y_true.values if isinstance(y_true, pd.Series) else y_true
    return np.sum(y_true == y_pred) / len(y_true)

def train_and_evaluate_rf(data, train_key_X, train_key_y, val_key_X, val_key_y, model_params):
    """Utility function to train and evaluate the RF on a specific dataset."""
    print(f"\n--- Training Parallel Random Forest on **{train_key_X.upper().replace('X_TRAIN_', '')}** Dataset ---")
    
    # Prepare data
    X_train = data[train_key_X]
    y_train = data[train_key_y]
    X_val = data[val_key_X]
    y_val = data[val_key_y]
    
    # Initialize and train the model
    rf = RandomForestClassifier(**model_params)
    start_time = time.time()
    rf.fit(X_train, y_train)
    end_time = time.time()
    
    print(f"\nTraining completed in **{end_time - start_time:.2f} seconds**.")
    
    # Make predictions
    y_train_pred = rf.predict(X_train)
    y_val_pred = rf.predict(X_val)
    
    # Evaluate
    train_acc = accuracy(y_train, y_train_pred)
    val_acc = accuracy(y_val, y_val_pred)
    
    print(f"**Training Accuracy:** {train_acc:.4f}")
    print(f"**Validation Accuracy:** {val_acc:.4f}")
    
    return rf, train_acc, val_acc

# Define model parameters
RF_PARAMS = {
    'n_estimators': 5,       # Reduce number of trees (10 instead of 50)
    'max_depth': 5,           # Shallower trees (5 instead of 10)
    'n_features_ratio': 0.3,  # Fewer features per split (0.3 instead of 0.5)
    'bootstrap_ratio': 0.7,   # Slightly smaller bootstrap sample
    'random_state': 42,
    'n_jobs': os.cpu_count()  # Keep using all CPU cores
}


# List of datasets to process
datasets = [
    ('X_train_smote', 'y_train_smote'),
    ('X_train_tomek', 'y_train_tomek'),
    ('X_train_smote_tomek', 'y_train_smote_tomek'),
]

results = {}
val_features_key = 'X_val'
val_target_key = 'y_val'

# Run the training loop for all sampled datasets
for X_key, y_key in datasets:
    model, train_acc, val_acc = train_and_evaluate_rf(
        data, 
        X_key, 
        y_key, 
        val_features_key, 
        val_target_key, 
        RF_PARAMS
    )
    # Store results
    dataset_name = X_key.replace('X_train_', '')
    results[dataset_name] = {
        'model': model,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc
    }

# --- Part 4: Final Summary ---

print("\n" + "="*50)
print("FINAL RANDOM FOREST TRAINING RESULTS (PARALLEL FROM SCRATCH)")
print("="*50)
for name, res in results.items():
    print(f"🌳 **{name.upper()}**:")
    print(f"  - Training Accuracy: {res['train_accuracy']:.4f}")
    print(f"  - Validation Accuracy: {res['val_accuracy']:.4f}")
    print("-" * 25)


--- Training Parallel Random Forest on **SMOTE** Dataset ---
Training 5 trees using 12 parallel processes...
  Max Depth: 5, Features per split: 19


--- Training Parallel Random Forest on **SMOTE** Dataset ---
Training 50 trees using 12 parallel processes...
  Max Depth: 10, Features per split: 32